In [1]:
import os
#os.chdir('../')

from medclip import MedCLIPModel, MedCLIPVisionModelViT
from medclip import MedCLIPProcessor
from medclip import PromptClassifier

/home/yuhaowang/anaconda3/envs/gigapath/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# init models
processor = MedCLIPProcessor()
model = MedCLIPModel(vision_cls=MedCLIPVisionModelViT, checkpoint='./data/checkpoint')
clf = PromptClassifier(model, ensemble=True)
clf.cuda()

/home/yuhaowang/anaconda3/envs/gigapath/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/yuhaowang/anaconda3/envs/gigapath/lib/python3.9/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Some weights of the model checkpoint at microsoft/swin-tiny-patch4-window7-224 were not used when initializing SwinModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing SwinModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassificatio

load model weight from: ./data/checkpoint


PromptClassifier(
  (model): MedCLIPModel(
    (vision_model): MedCLIPVisionModelViT(
      (model): SwinModel(
        (embeddings): SwinEmbeddings(
          (patch_embeddings): SwinPatchEmbeddings(
            (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
          )
          (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (encoder): SwinEncoder(
          (layers): ModuleList(
            (0): SwinStage(
              (blocks): ModuleList(
                (0-1): 2 x SwinLayer(
                  (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
                  (attention): SwinAttention(
                    (self): SwinSelfAttention(
                      (query): Linear(in_features=96, out_features=96, bias=True)
                      (key): Linear(in_features=96, out_features=96, bias=True)
                      (value): Linear(in_features=96, out_features=9

In [3]:
# prepare input image
from PIL import Image
image = Image.open('./example_data/view1_frontal.jpg')
inputs = processor(images=image, return_tensors="pt")

# prepare input prompt texts
from medclip.prompts import generate_chexpert_class_prompts, process_class_prompts

cls_prompts = process_class_prompts(generate_chexpert_class_prompts(n=10))
inputs['prompt_inputs'] = cls_prompts

sample 10 num of prompts for Atelectasis from total 210
sample 10 num of prompts for Cardiomegaly from total 15
sample 10 num of prompts for Consolidation from total 192
sample 10 num of prompts for Edema from total 18
sample 10 num of prompts for Pleural Effusion from total 54


In [4]:
output = clf(**inputs)
print(output)

{'logits': tensor([[0.3571, 0.4751, 0.1620, 0.2420, 0.3859]], device='cuda:0',
       grad_fn=<StackBackward0>), 'class_names': ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']}


In [1]:
# -*- coding: utf-8 -*-
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional
from PIL import Image
import torch
from medclip import PromptClassifier
from medclip import MedCLIPModel, MedCLIPVisionModelViT, MedCLIPProcessor
 # ← 替换成你的实际路径
from medclip.prompts import generate_chexpert_class_prompts, process_class_prompts

# -------------------------
# 模型与处理器
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
processor = MedCLIPProcessor()
model = MedCLIPModel(vision_cls=MedCLIPVisionModelViT)
model.from_pretrained()
model = model.to(DEVICE).eval()
clf = PromptClassifier(model, ensemble=True).to(DEVICE).eval()

# -------------------------
# PromptBank：一次生成，多图复用
# -------------------------
@dataclass
class PromptBank:
    raw_cls_prompts: Dict[str, List[str]]                         # {class: [prompt, ...]}
    prompt_map_by_class: Dict[str, Dict[str, Tuple[str,str,str]]] # {class: {prompt: (sev,sub,loc)}}
    cls_prompt_inputs: Dict[str, Dict[str, torch.Tensor]]         # tokenizer 结果（可直接复用）

def build_prompt_bank(
    n_prompts_per_class: int = 10,
    seed: Optional[int] = None
) -> PromptBank:
    raw_cls_prompts, prompt_map_by_class = generate_chexpert_class_prompts(
        n=n_prompts_per_class, return_map=True, seed=seed
    )
    cls_prompt_inputs = process_class_prompts(raw_cls_prompts)  
    return PromptBank(
        raw_cls_prompts=raw_cls_prompts,
        prompt_map_by_class=prompt_map_by_class,
        cls_prompt_inputs=cls_prompt_inputs
    )

# -------------------------
# 主推理：一张片 → K 个三元组（支持共享 PromptBank）
# -------------------------
@torch.inference_mode()
def infer_triples_via_best_prompt(
    image_path: str,
    k_findings: int = 5,
    threshold_finding: float = 0.35,
    prompt_bank: Optional[PromptBank] = None,
    n_prompts_per_class: int = 10,
    seed: Optional[int] = None,
) -> Dict[str, Any]:
    """
    用法：
      # 1) 先建一次 PromptBank
      bank = build_prompt_bank(n_prompts_per_class=10, seed=42)
      # 2) 多张图重复调用（复用 bank）
      out = infer_triples_via_best_prompt(img_path, prompt_bank=bank)

    如果未传 prompt_bank，则内部会临时构建一次（不利于多图效率，但兼容老用法）。
    """

    # 0) 准备 PromptBank（若未传则临时建一次）
    if prompt_bank is None:
        prompt_bank = build_prompt_bank(n_prompts_per_class=n_prompts_per_class, seed=seed)

    raw_cls_prompts = prompt_bank.raw_cls_prompts
    prompt_map_by_class = prompt_bank.prompt_map_by_class
    cls_prompt_inputs = prompt_bank.cls_prompt_inputs

    # 1) 读图
    image = Image.open(image_path).convert("RGB")

    # 2) 前向：类级 logits + 每类最佳 prompt 的索引（由已改造的 PromptClassifier 返回）
    inputs_img = processor(images=image, return_tensors="pt")
    inputs_img = {k: v.to(DEVICE) for k, v in inputs_img.items()}
    inputs_img["prompt_inputs"] = cls_prompt_inputs  # 直接复用（CPU → 在 classifier 内部 .cuda()）

    out = clf(**inputs_img, return_prompt_details=True)
    class_names: List[str] = out["class_names"]
    class_logits: torch.Tensor = out["logits"].squeeze(0)  # [C]
    p_finding_all = class_logits.tolist()

    # 3) Top-K（带阈值）
    cls_with_prob = list(zip(class_names, p_finding_all))
    cls_with_prob.sort(key=lambda x: x[1], reverse=True)
    filtered = [c for c in cls_with_prob if c[1] >= threshold_finding]
    selected = filtered[:k_findings] if filtered else cls_with_prob[:k_findings]

    # 可选：No Finding 的“独占”策略（按需保留或删除）
    if len(selected) >= 2 and selected[0][0].lower() in {"no finding", "no findings"}:
        if (selected[0][1] - selected[1][1]) > 0.20:
            selected = [selected[0]]

    # 4) 用“最佳 prompt 的索引”从 PromptBank 映射中直接取回三元组
    name_to_idx = {n: i for i, n in enumerate(class_names)}
    per_class_best_idx: List[int] = out["per_class_best_idx"]

    triples_out: List[Dict[str, Any]] = []
    for finding, p_f in selected:
        cls_i = name_to_idx[finding]
        best_idx = per_class_best_idx[cls_i]

        best_prompt_text = raw_cls_prompts[finding][best_idx]
        sev, sub, loc = prompt_map_by_class[finding][best_prompt_text]

        # 空串转 None（更干净）
        sev = sev if (sev and sev.strip()) else None
        sub = sub if (sub and sub.strip()) else None
        loc = loc if (loc and loc.strip()) else None

        triples_out.append({
            "finding": finding,
            "p_finding": float(p_f),
            "best_prompt": best_prompt_text,
            "severity": sev,
            "subtype": sub,
            "location": loc
        })

    # 5) 排序
    triples_out.sort(key=lambda x: x["p_finding"], reverse=True)

    return {
        "image": image_path,
        "topk_findings": [{"name": n, "p": float(p)} for (n, p) in selected],
        "triples": triples_out
    }

# -------------------------
# 示例：先建一次 PromptBank，多图复用
# -------------------------

bank = build_prompt_bank(n_prompts_per_class=10, seed=42)

images = [
    "./example_data/view1_frontal.jpg",
    #"./example_data/view2_frontal.jpg",
    # ...
]
for img in images:
    res = infer_triples_via_best_prompt(
        image_path=img,
        k_findings=3,
        threshold_finding=0.35,
        prompt_bank=bank,   # 复用
    )
    print(res)
       # print(json.dumps(res, indent=2, ensure_ascii=False))


/home/yuhaowang/anaconda3/envs/gigapath/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yuhaowang/anaconda3/envs/gigapath/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/yuhaowang/anaconda3/envs/gigapath/lib/python3.9/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Some weights of the model checkpoint at microsoft/swin-tiny-patch4-window7-2

load model weight from: ./pretrained/medclip-vit
sample 5 num of prompts for No Finding from total 5
sample 10 num of prompts for Enlarged Cardiomediastinum from total 20
sample 10 num of prompts for Cardiomegaly from total 20
sample 10 num of prompts for Lung Lesion from total 120
sample 10 num of prompts for Lung Opacity from total 100
sample 10 num of prompts for Edema from total 28
sample 10 num of prompts for Consolidation from total 100
sample 10 num of prompts for Pneumonia from total 80
sample 10 num of prompts for Atelectasis from total 90
sample 10 num of prompts for Pneumothorax from total 100
sample 10 num of prompts for Pleural Effusion from total 128
sample 10 num of prompts for Pleural Other from total 54
sample 10 num of prompts for Fracture from total 210
sample 10 num of prompts for Support Devices from total 60
{'image': './example_data/view1_frontal.jpg', 'topk_findings': [{'name': 'Support Devices', 'p': 1.1483012437820435}, {'name': 'Edema', 'p': 0.539619147777557

In [1]:
from transformers import AutoImageProcessor

/home/yuhaowang/anaconda3/envs/gigapath/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'AutoImageProcessor' from 'transformers' (/home/yuhaowang/anaconda3/envs/gigapath/lib/python3.9/site-packages/transformers/__init__.py)